# PipelineWatch-NG — Module 2: Preprocessing & Feature Engineering
**Goal:** Convert raw Sentinel-1 GRD tiles to calibrated sigma-nought backscatter, compute NDVI vegetation anomaly from Sentinel-2, cluster and score FIRMS fire detections by intensity.  
All outputs are GeoTIFF + GeoDataFrame files consumed by Module 3 (ML detection).


## 2.1 — Setup & mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
warnings.filterwarnings('ignore')

PROJECT_ROOT = '/content/drive/MyDrive/PipelineWatch_NG'
RAW_S1    = f'{PROJECT_ROOT}/data/raw/sentinel1'
RAW_FIRMS = f'{PROJECT_ROOT}/data/raw/firms'
RAW_S2    = f'{PROJECT_ROOT}/data/raw/sentinel2'
PROCESSED = f'{PROJECT_ROOT}/data/processed'
OUTPUTS   = f'{PROJECT_ROOT}/outputs'
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(OUTPUTS,   exist_ok=True)

with open(f'{PROJECT_ROOT}/config/aoi_niger_delta.geojson') as f:
    AOI = json.load(f)
AOI_BOUNDS = (5.8, 4.2, 7.2, 5.2)
print('✅ Environment ready.')


In [ ]:
!pip install rioxarray rasterio pyproj scikit-learn scipy folium geopandas -q
print('✅ Packages ready.')


## 2.2 — Sentinel-1 SAR: unpack & inventory .SAFE files

In [ ]:
import zipfile, glob

# Unzip any downloaded .zip tiles
zip_files = glob.glob(f'{RAW_S1}/*.zip')
for zf in zip_files:
    safe_name = os.path.basename(zf).replace('.zip', '.SAFE')
    out_dir   = f'{RAW_S1}/{safe_name}'
    if not os.path.exists(out_dir):
        print(f'Unzipping {os.path.basename(zf)} ...')
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(RAW_S1)
    else:
        print(f'Already unzipped: {safe_name}')

safe_dirs = glob.glob(f'{RAW_S1}/*.SAFE')
print(f'\n✅ SAFE directories available: {len(safe_dirs)}')
for s in safe_dirs:
    print(f'   {os.path.basename(s)}')


In [ ]:
# Parse SAFE metadata from folder names
# S1A_IW_GRDH_1SDV_20240115T054321_...
import re
from datetime import datetime

def parse_safe_meta(safe_path):
    name = os.path.basename(safe_path)
    parts = name.split('_')
    try:
        dt = datetime.strptime(parts[4], '%Y%m%dT%H%M%S')
    except:
        dt = None
    return {
        'path': safe_path,
        'satellite': parts[0],
        'mode': parts[1],
        'product': parts[2],
        'datetime': dt
    }

meta_rows = [parse_safe_meta(s) for s in safe_dirs]
df_meta   = pd.DataFrame(meta_rows)
print('Sentinel-1 SAFE inventory:')
print(df_meta[['satellite','mode','product','datetime']].to_string())


## 2.3 — SAR preprocessing: GRD → sigma-nought backscatter
> We use `rioxarray` + `rasterio` to read the GeoTIFF measurement files inside each .SAFE. Full SNAP-based calibration requires SNAP installation (optional — see note below).

In [ ]:
import rioxarray as rxr
import rasterio
import xarray as xr
from pathlib import Path

def load_sar_measurement(safe_path, polarisation='vv'):
    """
    Load the measurement GeoTIFF from a Sentinel-1 GRD .SAFE directory.
    Returns an xarray DataArray with CRS set.
    
    Note: This loads the raw DN (digital number) values.
    Full radiometric calibration to sigma0 (dB) requires SNAP or snappy.
    We apply the standard linear approximation: sigma0_dB = 10*log10(DN^2 / K)
    where K is the calibration constant (approx 83.0 for S1 IW GRD).
    """
    meas_dir = Path(safe_path) / 'measurement'
    tiff_files = list(meas_dir.glob(f'*{polarisation}*.tiff'))
    
    if not tiff_files:
        tiff_files = list(meas_dir.glob('*.tiff'))
    if not tiff_files:
        raise FileNotFoundError(f'No TIFF found in {meas_dir}')
    
    tiff_path = tiff_files[0]
    print(f'   Loading: {tiff_path.name}')
    
    da = rxr.open_rasterio(tiff_path, masked=True).squeeze()
    da = da.rio.set_crs('EPSG:4326', inplace=True) if da.rio.crs is None else da
    return da

def dn_to_sigma0_db(da, K=83.0):
    """Convert raw DN to sigma-nought in dB."""
    dn = da.values.astype(np.float32)
    dn = np.where(dn == 0, np.nan, dn)
    sigma0_linear = (dn ** 2) / K
    sigma0_db     = 10 * np.log10(np.maximum(sigma0_linear, 1e-10))
    result = da.copy(data=sigma0_db)
    result.attrs['long_name'] = 'sigma0_VV_dB'
    return result

if safe_dirs:
    print('Processing Sentinel-1 SAR tiles...')
    sar_layers = []
    for safe in safe_dirs[:2]:   # process first 2 scenes
        print(f'\n  {os.path.basename(safe)}')
        try:
            da_raw  = load_sar_measurement(safe, 'vv')
            da_sig0 = dn_to_sigma0_db(da_raw)
            sar_layers.append({'path': safe, 'sigma0': da_sig0})
            print(f'   ✅ Shape: {da_sig0.shape}, dtype: {da_sig0.dtype}')
            print(f'   σ0 range: {float(np.nanmin(da_sig0.values)):.1f} to {float(np.nanmax(da_sig0.values)):.1f} dB')
        except Exception as e:
            print(f'   ⚠️  Could not load: {e}')
    print(f'\n✅ {len(sar_layers)} SAR layer(s) processed')
else:
    print('⚠️  No SAFE dirs found. Run Module 1 to download SAR data.')
    print('    Generating SYNTHETIC SAR layer for demonstration...')
    import numpy as np
    # Synthetic sigma0 mimicking Niger Delta water/land backscatter
    np.random.seed(42)
    H, W = 512, 512
    sigma0_synth = np.random.normal(-12, 4, (H, W)).astype(np.float32)  # water background
    # Add dark patches (oil spills — low backscatter ~-25 dB)
    for _ in range(8):
        r, c = np.random.randint(50, H-50), np.random.randint(50, W-50)
        rr, cc = np.random.randint(15, 45), np.random.randint(20, 60)
        sigma0_synth[r:r+rr, c:c+cc] = np.random.normal(-25, 1.5, (rr, cc))
    sar_layers = [{'path': 'synthetic', 'sigma0': sigma0_synth, 'synthetic': True}]
    print('   ✅ Synthetic SAR layer created (512×512)')


## 2.4 — Oil spill dark-spot detection from sigma-nought

In [ ]:
from scipy.ndimage import label, binary_closing, binary_opening
from sklearn.preprocessing import StandardScaler

def detect_dark_spots(sigma0_arr, water_threshold=-18.0, spill_threshold=-22.0,
                      min_pixels=25):
    """
    Detect potential oil spill dark spots in SAR sigma0 imagery.
    
    Logic:
      - Water pixels: sigma0 < water_threshold (dB)
      - Dark anomaly: sigma0 < spill_threshold (dB)
      - Apply morphological cleaning (remove noise)
      - Label connected components
      - Filter by minimum size
    
    Returns binary mask and labeled array.
    """
    arr = sigma0_arr if isinstance(sigma0_arr, np.ndarray) else sigma0_arr.values
    
    # Water mask (pixels below water backscatter threshold)
    water_mask = arr < water_threshold
    
    # Dark anomaly mask (potential oil)
    dark_mask  = arr < spill_threshold
    dark_mask  = np.where(np.isnan(arr), False, dark_mask)
    
    # Morphological ops: close small holes, remove salt noise
    dark_clean = binary_closing(dark_mask,  structure=np.ones((3,3)))
    dark_clean = binary_opening(dark_clean, structure=np.ones((2,2)))
    
    # Label connected components
    labeled, n_features = label(dark_clean)
    
    # Filter: keep only blobs >= min_pixels
    filtered = np.zeros_like(labeled)
    blob_info = []
    for i in range(1, n_features + 1):
        blob = labeled == i
        size = blob.sum()
        if size >= min_pixels:
            filtered[blob] = i
            # Record centroid (pixel coords)
            ys, xs = np.where(blob)
            blob_info.append({
                'blob_id': i,
                'pixel_count': size,
                'mean_sigma0': float(arr[blob].mean()),
                'min_sigma0':  float(arr[blob].min()),
                'centroid_y':  int(ys.mean()),
                'centroid_x':  int(xs.mean()),
            })
    
    return water_mask, dark_clean, filtered, blob_info

# Run detection on first SAR layer
layer = sar_layers[0]
sigma0_arr = layer['sigma0'] if isinstance(layer['sigma0'], np.ndarray) else layer['sigma0'].values

water_mask, dark_mask, labeled_blobs, blob_info = detect_dark_spots(sigma0_arr)

df_blobs = pd.DataFrame(blob_info)
print(f'✅ Dark-spot detection complete')
print(f'   Water pixels       : {water_mask.sum():,}')
print(f'   Dark anomaly pixels: {dark_mask.sum():,}')
print(f'   Candidate blobs    : {len(df_blobs)}')
if not df_blobs.empty:
    print()
    print(df_blobs.sort_values("pixel_count", ascending=False).head(10).to_string(index=False))


In [ ]:
# Visualise SAR sigma0 + dark spots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Module 2 — SAR Sigma-nought & Oil Spill Dark Spots', fontsize=14)

ax = axes[0]
im = ax.imshow(sigma0_arr, cmap='gray', vmin=-30, vmax=0)
ax.set_title('Sigma-nought σ0 (dB)')
plt.colorbar(im, ax=ax, label='dB')

ax2 = axes[1]
ax2.imshow(sigma0_arr, cmap='gray', vmin=-30, vmax=0, alpha=0.6)
ax2.imshow(np.ma.masked_where(dark_mask == 0, dark_mask),
           cmap='autumn', alpha=0.7, vmin=0, vmax=1)
ax2.set_title('Dark anomalies detected')

ax3 = axes[2]
ax3.imshow(sigma0_arr, cmap='gray', vmin=-30, vmax=0, alpha=0.5)
blob_display = np.ma.masked_where(labeled_blobs == 0, labeled_blobs)
ax3.imshow(blob_display, cmap='tab10', alpha=0.8)
ax3.set_title(f'Candidate spill blobs (n={len(df_blobs)})')

plt.tight_layout()
fig_path = f'{OUTPUTS}/module2_sar_darkspots.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure saved → {fig_path}')


## 2.5 — FIRMS fire data: cluster & score illegal refineries

In [ ]:
import glob
from scipy.spatial.distance import cdist

firms_files = glob.glob(f'{RAW_FIRMS}/*.csv')
if firms_files:
    df_firms = pd.concat([pd.read_csv(f) for f in firms_files], ignore_index=True)
    df_firms.columns = [c.lower() for c in df_firms.columns]
    print(f'✅ Loaded {len(df_firms)} FIRMS fire detections from {len(firms_files)} files')
else:
    print('⚠️  No FIRMS CSV found. Generating synthetic fire data...')
    np.random.seed(7)
    n = 320
    # Cluster synthetic fires around known Niger Delta refinery hotspots
    centers = [(4.95, 6.85), (4.45, 6.10), (4.62, 6.43), (5.02, 6.70), (4.78, 6.55)]
    rows = []
    for lat0, lon0 in centers:
        n_c = np.random.randint(40, 80)
        for _ in range(n_c):
            rows.append({
                'latitude':   lat0 + np.random.normal(0, 0.06),
                'longitude':  lon0 + np.random.normal(0, 0.06),
                'frp':        np.random.exponential(35) + 5,
                'brightness': np.random.normal(340, 20),
                'acq_date':   '2024-01-15',
                'confidence': np.random.choice(['n','h','l'], p=[0.6,0.3,0.1])
            })
    df_firms = pd.DataFrame(rows)
    print(f'   ✅ Synthetic: {len(df_firms)} fire pixels, 5 cluster centres')
print(df_firms[['latitude','longitude','frp','confidence']].describe())


In [ ]:
from sklearn.cluster import DBSCAN

# DBSCAN spatial clustering of fire detections
# eps=0.05° (~5km), min_samples=5 → identifies persistent burn clusters
coords = df_firms[['latitude','longitude']].values
db = DBSCAN(eps=0.05, min_samples=5, algorithm='ball_tree', metric='haversine').fit(
    np.radians(coords))
df_firms['cluster_id'] = db.labels_

n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_noise    = (db.labels_ == -1).sum()
print(f'✅ DBSCAN clustering complete')
print(f'   Fire clusters found : {n_clusters}')
print(f'   Noise (isolated)    : {n_noise} pixels')

# Summarise each cluster
cluster_summary = []
for cid in sorted(set(db.labels_)):
    if cid == -1:
        continue
    sub = df_firms[df_firms['cluster_id'] == cid]
    frp_sum  = sub['frp'].sum()
    frp_mean = sub['frp'].mean()
    # Risk score: FRP sum weighted by confidence
    conf_map  = {'h': 1.0, 'n': 0.8, 'l': 0.5}
    conf_wt   = sub['confidence'].map(conf_map).fillna(0.8).mean()
    risk_score = min(100, (frp_sum / 50) * conf_wt)   # normalised 0-100
    cluster_summary.append({
        'cluster_id':  cid,
        'n_pixels':    len(sub),
        'centroid_lat': sub['latitude'].mean(),
        'centroid_lon': sub['longitude'].mean(),
        'frp_sum_mw':  round(frp_sum, 1),
        'frp_mean_mw': round(frp_mean, 1),
        'risk_score':  round(risk_score, 1),
        'likely_illegal': risk_score > 40,
    })

df_clusters = pd.DataFrame(cluster_summary).sort_values('risk_score', ascending=False)
print()
print('Fire cluster summary:')
print(df_clusters.to_string(index=False))


In [ ]:
# Save processed fire clusters
cluster_path = f'{PROCESSED}/firms_clusters.csv'
df_clusters.to_csv(cluster_path, index=False)

# Save enriched fire detections
firms_proc_path = f'{PROCESSED}/firms_processed.csv'
df_firms.to_csv(firms_proc_path, index=False)

# Visualise clusters
fig, ax = plt.subplots(figsize=(10, 8))
aoi_box = plt.Polygon(
    [[AOI_BOUNDS[0], AOI_BOUNDS[1]], [AOI_BOUNDS[2], AOI_BOUNDS[1]],
     [AOI_BOUNDS[2], AOI_BOUNDS[3]], [AOI_BOUNDS[0], AOI_BOUNDS[3]]],
    fill=False, edgecolor='#e63946', linewidth=2)
ax.add_patch(aoi_box)
ax.set_xlim(AOI_BOUNDS[0]-0.1, AOI_BOUNDS[2]+0.1)
ax.set_ylim(AOI_BOUNDS[1]-0.1, AOI_BOUNDS[3]+0.1)

# All fire pixels
sc = ax.scatter(df_firms['longitude'], df_firms['latitude'],
                c=df_firms['frp'], cmap='hot_r', s=15, alpha=0.5, vmin=0, vmax=150)
plt.colorbar(sc, ax=ax, label='FRP (MW)')

# Cluster centroids
for _, row in df_clusters.iterrows():
    color = '#e63946' if row['likely_illegal'] else '#2196f3'
    ax.scatter(row['centroid_lon'], row['centroid_lat'],
               s=row['risk_score']*3, c=color, alpha=0.9,
               edgecolors='black', linewidths=1.5, zorder=5)
    ax.annotate(f"Cluster {row['cluster_id']}\nRisk:{row['risk_score']:.0f}",
                (row['centroid_lon'], row['centroid_lat']),
                textcoords='offset points', xytext=(6,6), fontsize=7)

ax.set_title('FIRMS Fire Clusters — Niger Delta (red=likely illegal refinery)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
fig_path = f'{OUTPUTS}/module2_fire_clusters.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure saved → {fig_path}')


## 2.6 — Sentinel-2: NDVI vegetation anomaly
> Sentinel-2 optical detects vegetation die-off along pipeline corridors caused by chronic leaks.

In [ ]:
# If Sentinel-2 data is not yet downloaded, we generate a synthetic NDVI field.
# Download process mirrors Sentinel-1 (Module 1) — just change producttype to 'S2MSI2A'.
# Here we focus on the analytical pipeline so you can test end-to-end.

import glob as glb

s2_files = glb.glob(f'{RAW_S2}/**/*.tiff', recursive=True) +            glb.glob(f'{RAW_S2}/**/*.jp2',  recursive=True)

if s2_files:
    print(f'Found {len(s2_files)} Sentinel-2 band files. Loading B04 (Red) and B08 (NIR)...')
    # Load actual B04 / B08 bands and compute NDVI
    import rioxarray as rxr
    b4_files = [f for f in s2_files if 'B04' in f or 'B04_10m' in f]
    b8_files = [f for f in s2_files if 'B08' in f or 'B08_10m' in f]
    b4 = rxr.open_rasterio(b4_files[0]).squeeze().values.astype(np.float32)
    b8 = rxr.open_rasterio(b8_files[0]).squeeze().values.astype(np.float32)
    ndvi = (b8 - b4) / (b8 + b4 + 1e-6)
    ndvi = np.clip(ndvi, -1, 1)
    print(f'✅ NDVI computed. Shape: {ndvi.shape}')
else:
    print('⚠️  No Sentinel-2 files found. Generating synthetic NDVI...')
    np.random.seed(13)
    H, W = 512, 512
    # Healthy vegetation baseline
    ndvi = np.random.normal(0.55, 0.12, (H, W)).astype(np.float32)
    ndvi = np.clip(ndvi, 0.1, 0.9)
    # Simulate pipeline corridor (diagonal band of reduced NDVI)
    for i in range(H):
        j_start = max(0, int(W*0.3 + i*0.2) - 15)
        j_end   = min(W, j_start + 30)
        ndvi[i, j_start:j_end] = np.random.normal(0.22, 0.08, j_end-j_start)
    ndvi = np.clip(ndvi, -0.1, 0.9)
    print('   ✅ Synthetic NDVI created (pipeline corridor anomaly embedded)')


In [ ]:
# Compute NDVI anomaly: difference from baseline (rolling mean or historical mean)
# Simple approach: flag pixels > 2 std below local mean
from scipy.ndimage import uniform_filter

ndvi_smooth   = uniform_filter(ndvi, size=15)   # local 15px mean
ndvi_std_map  = np.std(ndvi) * np.ones_like(ndvi)
ndvi_anomaly  = ndvi_smooth - ndvi             # negative = lower than surroundings
anomaly_mask  = ndvi_anomaly > (0.15)          # more than 0.15 NDVI units below baseline

# Compute NDVI change severity index (0-100)
severity = np.clip((ndvi_anomaly / 0.4) * 100, 0, 100)

print(f'NDVI statistics:')
print(f'  Mean NDVI      : {ndvi.mean():.3f}')
print(f'  Anomaly pixels : {anomaly_mask.sum():,} ({anomaly_mask.mean()*100:.1f}%)')
print(f'  Max severity   : {severity.max():.1f}')

# Save processed NDVI arrays
np.save(f'{PROCESSED}/ndvi.npy',         ndvi)
np.save(f'{PROCESSED}/ndvi_anomaly.npy', ndvi_anomaly)
np.save(f'{PROCESSED}/ndvi_severity.npy',severity)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Module 2 — Sentinel-2 NDVI & Vegetation Anomaly', fontsize=13)
axes[0].imshow(ndvi,        cmap='RdYlGn', vmin=-0.2, vmax=0.9); axes[0].set_title('NDVI')
axes[1].imshow(ndvi_anomaly,cmap='RdBu_r', vmin=-0.3, vmax=0.3); axes[1].set_title('NDVI Anomaly')
im = axes[2].imshow(severity,cmap='YlOrRd', vmin=0,   vmax=100);  axes[2].set_title('Severity index')
plt.colorbar(im, ax=axes[2], label='Severity (0-100)')
plt.tight_layout()
fp = f'{OUTPUTS}/module2_ndvi_anomaly.png'
plt.savefig(fp, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ NDVI figure saved → {fp}')


## 2.7 — Save all processed outputs for Module 3

In [ ]:
# Save blob detections
blobs_path = f'{PROCESSED}/sar_darkspot_blobs.csv'
if not df_blobs.empty:
    df_blobs.to_csv(blobs_path, index=False)
    print(f'✅ SAR blobs saved → {blobs_path}')

# Save SAR sigma0 array
np.save(f'{PROCESSED}/sigma0_vv.npy', sigma0_arr)
print(f'✅ sigma0 array saved')

print()
print('='*52)
print('MODULE 2 — PREPROCESSING COMPLETE')
print('='*52)
print(f'  SAR dark blobs      : {len(df_blobs)}')
print(f'  Fire clusters       : {len(df_clusters)}')
print(f'  Likely illegal fires: {df_clusters["likely_illegal"].sum()}')
print(f'  NDVI anomaly pixels : {anomaly_mask.sum():,}')
print()
print('Next → Module 3: ML anomaly detection & classification')
